# Membership_v2 파생변수 생성

- **입력**: `260510_merged_v2/Membership_v2.csv` (23,343명)
- **관측창**: 가입일 기준 0~20일 고정 (21일 cutoff)
- **출력**: `260510_features/Membership_features.csv`
- **참고**: park.ingyeom, kim.kwangil, legacy v3, 어드바이저 문서

## 현재 → 최종 피처 수
- **현재**: 35개
- **최종**: 약 60개

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE     = Path('..').resolve()
DATA_IN  = BASE / '_data/02_interim/260510_merged_v2'
DATA_OUT = BASE / '_data/02_interim/260510_features'
DATA_OUT.mkdir(exist_ok=True)

CUTOFF_DAYS = 21  # 관측창 고정

mem = pd.read_csv(DATA_IN / 'Membership_v2.csv',   encoding='utf-8-sig')
vh  = pd.read_csv(DATA_IN / 'View_History_v2.csv', encoding='utf-8-sig').drop_duplicates()
um  = pd.read_csv(DATA_IN / 'User_Mapping_v2.csv', encoding='utf-8-sig').drop_duplicates()
mm  = pd.read_csv(DATA_IN / 'Movie_Master_v2.csv', encoding='utf-8-sig')

# Movie_Master 중복 제거 (MOVIE_NUM 기준)
mm = mm.drop_duplicates(subset='MOVIE_NUM', keep='first')

print(f'Membership:   {mem.shape}')
print(f'View_History: {vh.shape}')
print(f'User_Mapping: {um.shape}')
print(f'Movie_Master: {mm.shape} (중복제거 후)')
mem.head(3)

Membership:   (23343, 15)
View_History: (170594, 5)
User_Mapping: (23720, 2)
Movie_Master: (14018, 4) (중복제거 후)


,USER_KEY,product_code,price,billing_method,max_screen,is_promotion,is_churn_prevented,payment_device,is_user_verified,gender,age,reg_date,reg_hour,end_date,is_repurchase
0,7a6960912bebe03c6e4c770eb1aa91329c3497f18f90ca...,pk_1489,100.0,134,4.0,1,0,pc,1,F,20.0,2014-03-21,20,2014-04-21,1
1,4ec765db76545c1d6dda9f421590bf9d02f584009f8d92...,pk_1487,100.0,190,1.0,1,1,pc,1,F,25.0,2009-03-21,14,2009-04-21,0
2,01b16f9f7ff29b48b1ee0d1a89d1eb9662474e5eedb8c2...,pk_1488,100.0,180,2.0,1,0,android,1,F,20.0,2009-03-21,2,2009-04-21,0


## 1. Membership 기반 파생변수

In [2]:
df = mem.copy()
def decode_date(val):
    if pd.isna(val): return pd.NaT
    parts = str(val).strip().split('-')
    if len(parts) != 3: return pd.NaT
    first, month, year = parts
    day = first[-2:]
    try:
        return pd.to_datetime(f'{int(year)+2000:04d}-{int(month):02d}-{int(day):02d}')
    except:
        return pd.NaT

df['reg_date']    = df['reg_date'].map(decode_date)
df['end_date']    = df['end_date'].map(decode_date)
df['cutoff_date'] = df['reg_date'] + pd.to_timedelta(CUTOFF_DAYS - 1, unit='D')
print('날짜 변환 샘플:')
print(df[['reg_date','end_date','cutoff_date']].head(3))

# 날짜
df['duration_days']  = (df['end_date'] - df['reg_date']).dt.days
df['reg_hour_group'] = pd.cut(df['reg_hour'], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(float).astype('Int64')

# 가격 (price OR is_promotion 중 하나만 → price 사용)
df['is_usd']        = (df['price'] < 100).astype(int)
df['price_per_day'] = (df['price'] / df['duration_days'].replace(0, np.nan)).round(2)

# 기기/요금제
df['device_group'] = df['payment_device'].map({
    'ios':'mobile','android':'mobile','mobile':'mobile',
    'pc':'pc','smarttv':'tv','ott':'tv'
})
df['is_standard'] = (df['max_screen'] == 2).astype(int)
df['is_premium']  = (df['max_screen'] == 4).astype(int)

# 인구통계
df['age_group']  = pd.cut(df['age'], bins=[0,19,29,39,49,120], labels=[0,1,2,3,4]).astype(float).astype('Int64')
df['gender_enc'] = df['gender'].map({'M':1,'F':0}).fillna(2).astype(int)

# 교호작용 (legacy)
df['age_x_screen']         = df['age'] * df['max_screen']
df['verified_x_age']       = df['is_user_verified'].map({'Y':1,'N':0,'1':1,'0':0}).fillna(0) * df['age']
df['is_senior_unverified'] = ((df['age'] >= 50) & (df['is_user_verified'].isin(['N','0',0,'']))).astype(int)

print(f'Membership 파생변수: {len([c for c in df.columns if c not in mem.columns])}개 추가')
print('추가된 컬럼:', [c for c in df.columns if c not in mem.columns])

날짜 변환 샘플:
    reg_date   end_date cutoff_date
0 2021-03-14 2021-04-14  2021-04-03
1 2021-03-09 2021-04-09  2021-03-29
2 2021-03-09 2021-04-09  2021-03-29
Membership 파생변수: 13개 추가
추가된 컬럼: ['cutoff_date', 'duration_days', 'reg_hour_group', 'is_usd', 'price_per_day', 'device_group', 'is_standard', 'is_premium', 'age_group', 'gender_enc', 'age_x_screen', 'verified_x_age', 'is_senior_unverified']


## 2. View History 전처리 (관측창 적용)

In [3]:
# USER_KEY 매핑 + 영화 장르 붙이기
vh_key   = vh.merge(um, on='USER_NUM', how='left')
vh_movie = vh_key.merge(mm[['MOVIE_NUM','genre','ott_release_month']], on='MOVIE_NUM', how='left')

# 날짜 변환
vh_movie['watch_date'] = pd.to_datetime(
    vh_movie['watch_day'].fillna(0).astype(int).astype(str),
    format='%Y%m%d', errors='coerce'
)

# reg_date, cutoff_date 매핑
df_dedup = df.drop_duplicates(subset='USER_KEY', keep='first')
reg_map    = df_dedup.set_index('USER_KEY')['reg_date']
cutoff_map = df_dedup.set_index('USER_KEY')['cutoff_date']
end_map    = df_dedup.set_index('USER_KEY')['end_date']

vh_movie['reg_date_ref']   = pd.to_datetime(vh_movie['USER_KEY'].map(reg_map))
vh_movie['cutoff_date_ref']= pd.to_datetime(vh_movie['USER_KEY'].map(cutoff_map))

# ★ 관측창 적용: reg_date <= watch_date <= cutoff_date
vh_filtered = vh_movie[
    (vh_movie['watch_date'] >= vh_movie['reg_date_ref']) &
    (vh_movie['watch_date'] <= vh_movie['cutoff_date_ref'])
].copy()

vh_filtered['days_since_reg'] = (vh_filtered['watch_date'] - vh_filtered['reg_date_ref']).dt.days
vh_filtered['is_weekend']     = vh_filtered['watch_date'].dt.dayofweek.isin([5,6])
vh_filtered['obs_week']       = pd.cut(vh_filtered['days_since_reg'], bins=[-1,6,13,20], labels=[1,2,3]).astype(float).fillna(0).astype(int)
vh_filtered['is_new_movie']   = (vh_filtered['ott_release_month'] == 202103).astype(int)
vh_filtered['is_short_watch'] = (vh_filtered['watch_time(min)'] <= 5).astype(int)
vh_filtered['is_1min_watch']  = (vh_filtered['watch_time(min)'] == 1).astype(int)
vh_filtered['is_last7d']      = (vh_filtered['days_since_reg'] >= CUTOFF_DAYS - 7).astype(int)

print(f'원본 로그: {len(vh_movie):,}행 → 관측창 적용 후: {len(vh_filtered):,}행')
print(f'제외된 로그: {len(vh_movie)-len(vh_filtered):,}행')

원본 로그: 170,594행 → 관측창 적용 후: 151,185행
제외된 로그: 19,409행


## 3. 유저별 피처 집계

In [4]:
rows = []
for user_key, grp in vh_filtered.groupby('USER_KEY'):
    row = {'USER_KEY': user_key}
    dur_days = max((pd.to_datetime(end_map.get(user_key, pd.NaT)) -
                    pd.to_datetime(reg_map.get(user_key, pd.NaT))).days, 1)
    cutoff_dt = pd.to_datetime(cutoff_map.get(user_key, pd.NaT))

    # ── 기본 집계 ─────────────────────────────────
    row['total_sessions']   = len(grp)
    row['unique_movies']    = grp['MOVIE_NUM'].nunique()
    row['active_days']      = grp['watch_date'].nunique()
    row['total_watch_time'] = grp['watch_time(min)'].sum()
    row['activity_rate']    = round(row['active_days'] / CUTOFF_DAYS, 4)
    row['watch_per_day']    = round(row['total_sessions'] / max(row['active_days'],1), 4)
    row['avg_rewatch_ratio']= round((row['total_sessions']-row['unique_movies']) / max(row['total_sessions'],1), 4)
    row['has_watch_history']= 1

    # ── 세션 통계 (park.ingyeom) ──────────────────
    row['avg_session_time']    = round(grp['watch_time(min)'].mean(), 4)
    row['median_session_time'] = round(grp['watch_time(min)'].median(), 4)
    row['std_session_time']    = round(grp['watch_time(min)'].std(skipna=True) or 0, 4)
    row['min_session_time']    = grp['watch_time(min)'].min()
    row['max_session_time']    = grp['watch_time(min)'].max()

    # ── 영화당 평균 시청 (park.ingyeom) ───────────
    row['avg_watch_time_per_content'] = round(row['total_watch_time'] / max(row['unique_movies'],1), 4)

    # ── 일별 집계 (park.ingyeom) ──────────────────
    daily_wt = grp.groupby('watch_date')['watch_time(min)'].sum()
    daily_sc = grp.groupby('watch_date').size()
    row['avg_daily_watch_time']    = round(daily_wt.mean(), 4)
    row['median_daily_watch_time'] = round(daily_wt.median(), 4)
    row['std_daily_watch_time']    = round(daily_wt.std(skipna=True) or 0, 4)
    row['max_daily_watch_time']    = daily_wt.max()
    row['max_day_share']           = round(row['max_daily_watch_time'] / max(row['total_watch_time'],1e-9), 4)
    row['max_daily_sessions']      = int(daily_sc.max())
    row['contents_per_active_day'] = round(row['unique_movies'] / max(row['active_days'],1), 4)

    # ── 온보딩 ────────────────────────────────────
    first_day = grp['days_since_reg'].min()
    last_day  = grp['days_since_reg'].max()
    row['cold_start']            = int(first_day <= 7) if pd.notna(first_day) else 0
    row['signup_to_first_watch'] = int(first_day) if pd.notna(first_day) else CUTOFF_DAYS
    row['active_span_days']      = int(last_day - first_day) if pd.notna(first_day) and pd.notna(last_day) else 0

    # ── recency: cutoff 기준 ──────────────────────
    last_date = grp['watch_date'].max()
    row['recency'] = int((cutoff_dt - last_date).days) if pd.notna(last_date) else CUTOFF_DAYS

    # ── 공백 ──────────────────────────────────────
    row['sessions_per_active_day'] = round(row['total_sessions'] / max(row['active_days'],1), 4)
    sorted_days = sorted(grp['watch_date'].dropna().unique())
    if len(sorted_days) >= 2:
        gaps = [(sorted_days[i+1]-sorted_days[i]).days for i in range(len(sorted_days)-1)]
        row['max_inactive_gap_days']  = max(gaps)
        row['avg_gap_between_watch_days'] = round(sum(gaps)/len(gaps), 4)
    else:
        row['max_inactive_gap_days']      = 0
        row['avg_gap_between_watch_days'] = 0

    # ── 짧은 시청 ─────────────────────────────────
    row['vh_short_watch_ratio']   = round(grp['is_short_watch'].sum() / max(row['total_sessions'],1), 4)
    row['vh_one_min_watch_ratio'] = round(grp['is_1min_watch'].sum()  / max(row['total_sessions'],1), 4)
    row['vh_last7d_watch_ratio']  = round(grp['is_last7d'].sum()      / max(row['total_sessions'],1), 4)
    row['vh_title_div_per_day']   = round(row['unique_movies'] / max(row['active_days'],1), 4)
    row['weekend_watch_ratio']    = round(grp['is_weekend'].mean(), 4)

    # ── 주차별 ────────────────────────────────────
    for w in [1,2,3]:
        wg = grp[grp['obs_week']==w]
        row[f'dur_w{w}']         = wg['watch_time(min)'].sum()
        row[f'week{w}_sessions'] = len(wg)
        row[f'week{w}_active']   = wg['watch_date'].nunique()

    row['retention_w2'] = int(row['dur_w2'] > 0)
    row['retention_w3'] = int(row['dur_w3'] > 0)
    row['retention_w2_ratio'] = round(row['dur_w2'] / (row['dur_w1']+1e-6), 4)
    row['retention_w3_ratio'] = round(row['dur_w3'] / (row['dur_w2']+1e-6), 4)

    # ── 주차 비율 (park.ingyeom) ──────────────────
    total_w = row['dur_w1'] + row['dur_w2'] + row['dur_w3']
    row['week1_ratio'] = round(row['dur_w1'] / (total_w+1e-6), 4)
    row['week2_ratio'] = round(row['dur_w2'] / (total_w+1e-6), 4)
    row['week3_ratio'] = round(row['dur_w3'] / (total_w+1e-6), 4)
    w3_w1_raw = row['dur_w3'] / (row['dur_w1']+1e-6)
    row['w3_to_w1_ratio_capped']  = round(min(w3_w1_raw, 999), 4)
    row['week_count_with_watch']  = int(row['dur_w1']>0) + int(row['dur_w2']>0) + int(row['dur_w3']>0)

    # ── 트렌드 ────────────────────────────────────
    row['w2_minus_w1']       = row['dur_w2'] - row['dur_w1']
    row['w3_minus_w2']       = row['dur_w3'] - row['dur_w2']
    row['w3_minus_w1']       = row['dur_w3'] - row['dur_w1']
    row['daily_watch_slope'] = round(row['w3_minus_w1'] / 14, 4)
    row['front_loaded_flag'] = int(row['week1_ratio'] >= 0.5)
    row['late_binge_flag']   = int(row['week3_ratio'] >= 0.5)
    row['steady_3week_flag'] = int(row['week_count_with_watch'] == 3)

    # ── 패턴 플래그 (park.ingyeom) ───────────────
    row['only_week1_flag'] = int(row['dur_w1']>0 and row['dur_w2']==0 and row['dur_w3']==0)
    row['only_week3_flag'] = int(row['dur_w1']==0 and row['dur_w2']==0 and row['dur_w3']>0)
    row['no_week1_flag']   = int(row['dur_w1'] == 0)
    row['no_week3_flag']   = int(row['dur_w3'] == 0)

    # ── 몰아보기 ──────────────────────────────────
    row['one_day_binge_flag'] = int(row['max_day_share'] >= 0.8) if row['total_watch_time'] > 0 else 0
    row['binge_day_count']    = int((daily_sc >= 3).sum())

    # ── 신작 / 출시연도 ────────────────────────────
    row['is_new_movie_ratio'] = round(grp['is_new_movie'].mean(), 4)
    rel_months = grp['ott_release_month'].dropna()
    if len(rel_months) > 0:
        years = rel_months.astype(str).str[:4].astype(float)
        row['avg_release_year'] = round((years * grp.loc[years.index, 'watch_time(min)']).sum() /
                                        max(grp.loc[years.index, 'watch_time(min)'].sum(), 1), 2)
    else:
        row['avg_release_year'] = 0

    # ── 교호작용 ──────────────────────────────────
    row['stream_watch_interaction'] = df_dedup.set_index('USER_KEY')['max_screen'].get(user_key, 1) * row['total_watch_time']

    # ── 장르 ──────────────────────────────────────
    genre_total = grp['genre'].notna().sum()
    for genre_name, pattern in [
        ('horror',   'Horror'),
        ('family',   'Animation/Family'),
        ('drama',    'Drama'),
        ('action',   'Action/Adventure'),
        ('thriller', 'Thriller/Crime'),
        ('sf',       'SF/Fantasy'),
        ('comedy',   'Comedy'),
        ('romance',  'Romance'),
    ]:
        row[f'{genre_name}_ratio'] = round(
            grp['genre'].dropna().str.contains(pattern).sum() / max(genre_total,1), 4
        )

    # 복합 장르 플래그 (park.ingyeom)
    row['action_sf_thriller_affinity'] = int((row['action_ratio']+row['sf_ratio']+row['thriller_ratio']) >= 0.4)
    row['family_content_affinity']     = int(row['family_ratio'] >= 0.1)

    # 장르 엔트로피
    if genre_total > 0:
        gc = grp['genre'].value_counts(normalize=True)
        entropy = -(gc * np.log(gc+1e-9)).sum()
        max_ent = np.log(len(gc)) if len(gc) > 1 else 1
        row['genre_entropy_norm'] = round(entropy / max_ent, 4)
    else:
        row['genre_entropy_norm'] = 0

    rows.append(row)

if rows:
    vh_feat = pd.DataFrame(rows)
else:
    print('경고: 시청 이력 데이터 없음')
    vh_feat = pd.DataFrame({'USER_KEY': []})
print(f'View History 파생변수 완료: {vh_feat.shape[1]-1}개 피처 / {len(vh_feat):,}명')
vh_feat.head(3)

View History 파생변수 완료: 78개 피처 / 21,699명


,USER_KEY,total_sessions,unique_movies,active_days,total_watch_time,activity_rate,watch_per_day,avg_rewatch_ratio,has_watch_history,avg_session_time,...,family_ratio,drama_ratio,action_ratio,thriller_ratio,sf_ratio,comedy_ratio,romance_ratio,action_sf_thriller_affinity,family_content_affinity,genre_entropy_norm
0,0000555c21e7942b8281c8068c2b5be0a628b8a1a3cbea...,15,7,10,445,0.4762,1.5000,0.5333,1,29.6667,...,0.0,0.1333,0.00,0.8667,0.00,0.0,0.0,1,0,0.5665
1,0006075c3c18078eb09940cd27c6359a96a2a17fce8055...,5,4,3,452,0.1429,1.6667,0.2000,1,90.4000,...,0.8,0.0000,0.00,0.0000,0.00,0.0,0.0,0,1,0.7219
2,0008a7c034e8493a4b98463eddd53fd39de41f820dbd76...,4,4,4,111,0.1905,1.0000,0.0000,1,27.7500,...,0.0,0.5000,0.25,0.0000,0.25,0.0,0.0,1,0,0.9464


## 4. 합치고 저장

In [5]:
result = df.merge(vh_feat, on='USER_KEY', how='left')

vh_cols = [
    # 기본
    'total_sessions','unique_movies','active_days','total_watch_time',
    'activity_rate','watch_per_day','avg_rewatch_ratio','has_watch_history',
    # 세션 통계
    'avg_session_time','median_session_time','std_session_time','min_session_time','max_session_time',
    'avg_watch_time_per_content',
    # 일별
    'avg_daily_watch_time','median_daily_watch_time','std_daily_watch_time',
    'max_daily_watch_time','max_day_share','max_daily_sessions','contents_per_active_day',
    # 온보딩/리텐션
    'cold_start','signup_to_first_watch','recency','active_span_days',
    'sessions_per_active_day','max_inactive_gap_days','avg_gap_between_watch_days',
    # 짧은 시청
    'vh_short_watch_ratio','vh_one_min_watch_ratio','vh_last7d_watch_ratio','vh_title_div_per_day',
    'weekend_watch_ratio',
    # 주차별
    'dur_w1','dur_w2','dur_w3','week1_sessions','week2_sessions','week3_sessions',
    'week1_active','week2_active','week3_active',
    'retention_w2','retention_w3','retention_w2_ratio','retention_w3_ratio',
    # 주차 비율/트렌드
    'week1_ratio','week2_ratio','week3_ratio','w3_to_w1_ratio_capped','week_count_with_watch',
    'w2_minus_w1','w3_minus_w2','w3_minus_w1','daily_watch_slope',
    # 패턴 플래그
    'front_loaded_flag','late_binge_flag','steady_3week_flag',
    'only_week1_flag','only_week3_flag','no_week1_flag','no_week3_flag',
    # 몰아보기
    'one_day_binge_flag','binge_day_count',
    # 콘텐츠
    'is_new_movie_ratio','avg_release_year','stream_watch_interaction',
    # 장르
    'horror_ratio','family_ratio','drama_ratio','action_ratio','thriller_ratio',
    'sf_ratio','comedy_ratio','romance_ratio',
    'action_sf_thriller_affinity','family_content_affinity','genre_entropy_norm',
]
result[vh_cols] = result[vh_cols].fillna(0)
result.loc[result['has_watch_history']==0, 'recency'] = CUTOFF_DAYS

# 불필요 컬럼 제거
DROP = ['max_screen','reg_date','end_date','reg_hour','cutoff_date','is_promotion']
result = result.drop(columns=[c for c in DROP if c in result.columns])

out_path = DATA_OUT / 'Membership_features.csv'
result.to_csv(out_path, index=False, encoding='utf-8-sig')

orig_cols  = len(mem.columns)
total_feat = len(vh_cols) + len([c for c in df.columns if c not in mem.columns and c not in DROP])
print(f"저장 완료: {out_path}")
print(f"원본 컬럼:  {orig_cols}개")
print(f"파생변수:   {total_feat}개")
print(f"최종 shape: {result.shape}  ({result.shape[1]}개 컬럼)")
print()
print("전체 컬럼:", list(result.columns))

저장 완료: C:\Users\USER\OneDrive\바탕 화면\AX git\ott-churn-prediction\kwon.donggeun\_data\02_interim\260510_features\Membership_features.csv
원본 컬럼:  15개
파생변수:   90개
최종 shape: (23343, 100)  (100개 컬럼)

전체 컬럼: ['USER_KEY', 'product_code', 'price', 'billing_method', 'is_churn_prevented', 'payment_device', 'is_user_verified', 'gender', 'age', 'is_repurchase', 'duration_days', 'reg_hour_group', 'is_usd', 'price_per_day', 'device_group', 'is_standard', 'is_premium', 'age_group', 'gender_enc', 'age_x_screen', 'verified_x_age', 'is_senior_unverified', 'total_sessions', 'unique_movies', 'active_days', 'total_watch_time', 'activity_rate', 'watch_per_day', 'avg_rewatch_ratio', 'has_watch_history', 'avg_session_time', 'median_session_time', 'std_session_time', 'min_session_time', 'max_session_time', 'avg_watch_time_per_content', 'avg_daily_watch_time', 'median_daily_watch_time', 'std_daily_watch_time', 'max_daily_watch_time', 'max_day_share', 'max_daily_sessions', 'contents_per_active_day', 'cold_start',